# 2. KVLCC2: surface flow, integral boundary layer, and the viscous correction

Notebook 1 established that the zero-frequency potential flow gives
$Y_v \equiv 0$ — ideal flow produces no lateral damping at all. Every
maneuvering method therefore needs a mechanism to supply it. Wang et al. use
the semi-empirical Schmitz stern truncation. This notebook documents the
alternative that was added to MarineHydro: a **quasi-three-dimensional Head
integral boundary layer** marched over the hull, linearized with respect to
sway velocity and yaw rate, and fed back through the same boundary-element
operator.

The test case is the one the Gothenburg 2010 CFD workshop defined for KVLCC2:
original hull, bare, double body, $Re = 4.6\times10^{6}$, $Fn = 0.142$. The
geometry is the public workshop surface definition, fetched by
`validation/gothenburg2010/fetch_geometry.sh` into an untracked `data/`
directory.

**Read notebook 1 first** — the boundary-value problem, the derivative
integrals and their verification are established there.

In [ ]:
using Pkg

function marinehydro_root(start=pwd())
    directory = abspath(start)
    while true
        project = joinpath(directory, "Project.toml")
        if isfile(project) && occursin("name = \"MarineHydro\"", read(project, String))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Run Jupyter from the MarineHydro.jl repository.")
        directory = parent
    end
end

project_root = marinehydro_root()
Pkg.activate(project_root)

using Revise
using MarineHydro
using CairoMakie
using TOML
using LinearAlgebra: norm
CairoMakie.activate!()

validation_directory = joinpath(project_root, "validation", "gothenburg2010")
results_directory = joinpath(validation_directory, "results")
mkpath(results_directory)

data_directory = joinpath(validation_directory, "data", "KVLCC2")
surface_paths = [
    joinpath(data_directory, "kvlcc_bow1.dat"),
    joinpath(data_directory, "kvlcc2_stn1.dat"),
]
all(isfile, surface_paths) ||
    error("Run validation/gothenburg2010/fetch_geometry.sh first.")

# Workshop condition. The imported geometry is normalised by Lpp, so L = 1 and
# the Froude number fixes the speed directly.
REYNOLDS_NUMBER = 4.6e6
FROUDE_NUMBER = 0.142
WATER_DENSITY = 1025.0
LENGTH_PP = 320.0
DRAFT = 20.8

forward_speed = FROUDE_NUMBER * sqrt(SETTINGS.g)
kinematic_viscosity = forward_speed / REYNOLDS_NUMBER
(; forward_speed, kinematic_viscosity)

## 2.1 Geometry: structured hull patches and the panel grid

The boundary layer needs more than a bag of panels. Marching an integral
boundary layer requires an ordered sequence of stations along the flow, so the
importer preserves the **topology** of the workshop's structured surface
patches. `read_gothenburg2010_panel_grid` returns a `StructuredPanelGrid`,
which is a `Mesh` plus a list of `strips`: each strip is a vector of panel
indices running from bow to stern along one girthwise station line.

The importer performs four operations that are worth naming, because they are
the parts most likely to be wrong in a hull importer:

1. **Patch assembly.** The bow and after-body patches are read as separate
   Tecplot-format structured surfaces and joined into one longitudinal grid.
2. **Waterline clipping.** Panels are cut exactly at $z=0$ so the immersed
   surface is closed by the waterplane, rather than clipped at panel
   boundaries.
3. **Mirroring.** The workshop defines one side; the port side is generated by
   reflection, doubling the panel count and giving a starboard–port symmetric
   mesh.
4. **Resampling.** `target_shape = (n_longitudinal, n_girth)` resamples the
   structured surface, which is how the convergence sequences below are built.

Two integral quantities check that all four steps are right, because both are
published for KVLCC2: the wetted surface $S/L^2$ and the displaced volume
$\nabla/L^3$.

In [ ]:
reference_area = 27194.0 / LENGTH_PP^2
reference_volume = 312622.0 / LENGTH_PP^3

geometry_rows = map(((8, 5), (10, 6), (12, 7), (16, 9), (20, 12), (40, 24), nothing)) do shape
    mesh = read_gothenburg2010_mesh(surface_paths; target_shape=shape)
    area = mesh_surface_area(mesh)
    volume = mesh_signed_volume(mesh)
    (; shape=isnothing(shape) ? "native" : "$(shape[1])x$(shape[2])",
       panels=mesh.nfaces,
       area, area_error=area / reference_area - 1,
       volume, volume_error=volume / reference_volume - 1)
end
for row in geometry_rows
    println(row)
end

## 2.2 The double-body surge flow

The boundary layer is driven by the inviscid velocity just outside it. In the
body frame that is

$$\boldsymbol{u}_e = U\nabla\phi_1 + v\nabla\phi_v + r\nabla\phi_r
- \big[U\boldsymbol{e}_x + v\boldsymbol{e}_y
+ r\,\boldsymbol{e}_z\times(\boldsymbol{x}-\boldsymbol{x}_0)\big],$$

assembled by `body_relative_edge_velocity`. The bracket is the rigid-body
velocity of the hull surface itself; subtracting it converts the earth-frame
potential gradient into the velocity the fluid has *relative to the wall*,
which is what a boundary layer responds to. The result is then projected with
$(\boldsymbol{I}-\boldsymbol{n}\boldsymbol{n}^{\mathsf T})$ to remove the
panel-quadrature residual in the normal direction.

At $v = r = 0$ this is pure surge. Note the sign: with the bow at positive $x$,
the incoming stream in the body frame runs toward **negative** $x$, so
$\boldsymbol{u}_e$ points aft over most of the hull and the surface streamlines
below run bow to stern. `surface_stagnation_panel` locates the lowest-speed
panel in the forebody, and `trace_surface_streamlines` integrates the
tangential field across panel neighbours.

In [ ]:
visualization_shape = (32, 17)
visualization_grid = read_gothenburg2010_panel_grid(
    surface_paths; target_shape=visualization_shape,
)
visualization_mesh = visualization_grid.mesh
visualization_surge = solve_rigid_body_potential(visualization_mesh, :surge)
visualization_edge_velocity = body_relative_edge_velocity(
    visualization_mesh, 1.0, 0.0, 0.0;
    surge_gradient=visualization_surge.potential_gradient,
)
visualization_speed = [
    norm(@view visualization_edge_velocity[panel, :])
    for panel in 1:visualization_mesh.nfaces
]

longitudinal_extent = -(-(extrema(visualization_mesh.centers[:, 1])...))
bow_limit = maximum(visualization_mesh.centers[:, 1]) - 0.12 * longitudinal_extent
stagnation = surface_stagnation_panel(
    visualization_mesh, visualization_edge_velocity;
    panel_mask=visualization_mesh.centers[:, 1] .>= bow_limit,
)
streamlines = trace_surface_streamlines(
    visualization_mesh, visualization_edge_velocity,
    [first(strip) for strip in visualization_grid.strips];
    step_size=0.003 * longitudinal_extent,
    max_steps=650,
    neighbor_count=10,
    maximum_surface_distance=0.06 * longitudinal_extent,
)

(; panels=visualization_mesh.nfaces,
   strips=length(visualization_grid.strips),
   stagnation_point=stagnation.point,
   stagnation_speed_ratio=stagnation.speed)

In [ ]:
function panel_wireframe(mesh)
    x = Float64[]; y = Float64[]; z = Float64[]
    for panel in 1:mesh.nfaces
        indices = mesh.faces[panel, :] .+ 1
        for index in (indices[1], indices[2], indices[3], indices[4], indices[1])
            push!(x, mesh.vertices[index, 1])
            push!(y, mesh.vertices[index, 2])
            push!(z, mesh.vertices[index, 3])
        end
        push!(x, NaN); push!(y, NaN); push!(z, NaN)
    end
    return x, y, z
end

wire_x, wire_y, wire_z = panel_wireframe(visualization_mesh)
x_bounds = extrema(visualization_mesh.vertices[:, 1])
y_bounds = extrema(visualization_mesh.vertices[:, 2])
z_bounds = extrema(visualization_mesh.vertices[:, 3])

function add_hull_flow!(axis; x_limits=x_bounds, show_stagnation=false)
    lines!(axis, wire_x, wire_y, wire_z; color=(:gray25, 0.28), linewidth=0.45)
    scatter!(axis, visualization_mesh.centers[:, 1],
        visualization_mesh.centers[:, 2], visualization_mesh.centers[:, 3];
        color=visualization_speed, colormap=:viridis, colorrange=(0, 1.35),
        markersize=3.0)
    for line in streamlines
        size(line.points, 1) < 2 && continue
        lines!(axis, line.points[:, 1], line.points[:, 2], line.points[:, 3];
            color=:orangered2, linewidth=1.8)
        scatter!(axis, [line.points[1, 1]], [line.points[1, 2]], [line.points[1, 3]];
            color=:gold, markersize=4)
    end
    if show_stagnation
        scatter!(axis, [stagnation.point[1]], [stagnation.point[2]],
            [stagnation.point[3]];
            color=:red, marker=:star5, markersize=18,
            strokecolor=:white, strokewidth=1)
    end
    xlims!(axis, x_limits...)
    ylims!(axis, y_bounds...)
    zlims!(axis, 1.12 * z_bounds[1], 0.012)
    return axis
end

figure = Figure(size=(1800, 650), backgroundcolor=:white)
full_axis = Axis3(figure[1, 1]; title="Full hull: bow is +x, flow runs toward −x",
    xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp", aspect=:data,
    azimuth=1.18pi, elevation=0.16pi)
bow_axis = Axis3(figure[1, 2]; title="Bow stagnation region",
    xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp", aspect=:data,
    azimuth=1.20pi, elevation=0.13pi)
stern_axis = Axis3(figure[1, 3]; title="Stern surface streamlines",
    xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp", aspect=:data,
    azimuth=1.20pi, elevation=0.13pi)

add_hull_flow!(full_axis; show_stagnation=true)
add_hull_flow!(bow_axis;
    x_limits=(x_bounds[2] - 0.18 * longitudinal_extent, x_bounds[2]),
    show_stagnation=true)
add_hull_flow!(stern_axis;
    x_limits=(x_bounds[1], x_bounds[1] + 0.20 * longitudinal_extent))

Colorbar(figure[1, 4]; limits=(0, 1.35), colormap=:viridis, label="|u_e| / U")
Label(figure[0, :],
    "KVLCC2 unrestricted infinite-depth double-body surface flow, $(visualization_mesh.nfaces) panels";
    fontsize=24, font=:bold)
save(joinpath(results_directory, "kvlcc2_surface_flow.png"), figure; px_per_unit=1.5)
figure

## 2.3 Head's turbulent integral boundary layer

Rather than resolving the boundary layer, an integral method tracks two
thickness parameters along the flow. Define the momentum thickness $\theta$,
displacement thickness $\delta^{*}$ and shape factor $H=\delta^{*}/\theta$.
The von Kármán momentum-integral equation and Head's entrainment equation are

$$\frac{d\theta}{ds} + (H+2)\frac{\theta}{U_e}\frac{dU_e}{ds} = \frac{C_f}{2},
\qquad
\frac{d}{ds}\big(U_e\theta H_1\big) = U_e E .$$

Two equations, three unknowns $(\theta, H, H_1)$ — so the system is closed with
empirical correlations, all in `src/boundary_layers.jl`:

| Closure | Expression | Function |
|---|---|---|
| Entrainment shape parameter | $H_1 = 3.3 + 0.8234(H-1.1)^{-1.287}$ for $H\le1.6$, else $3.3 + 1.5501(H-0.6778)^{-3.064}$ | `head_kinetic_shape_factor` |
| Skin friction (Ludwieg–Tillmann) | $C_f = 0.246\,\cdot 10^{-0.678H}\,Re_\theta^{-0.268}$ | `head_skin_friction_coefficient` |
| Entrainment rate (Head) | $E = 0.0306\,(H_1-3)^{-0.6169}$ | `head_entrainment_coefficient` |

Substituting $H_1(H)$ turns the pair into two coupled ODEs in $(\theta, H)$,
marched by `solve_head_boundary_layer` with a fourth-order Runge–Kutta scheme
and sub-stepping between panel stations. The first station is initialised with
the turbulent flat-plate estimate $\theta = 0.036\,s\,Re_s^{-1/5}$; the layer
is assumed fully turbulent from the bow, which is standard for a ship at
$Re\sim10^{6}$–$10^{9}$ with a tripped or naturally turbulent forebody.

The layer is flagged **separated** when $H$ reaches $2.4$. That is a
*validity flag*, not a solution: Head's closure is an attached-flow
correlation, so past that point the integral state is frozen and the wall
shear set to zero. Nothing downstream of separation is modelled.

### Verification: does the march reproduce the flat-plate laws?

With $U_e$ constant the equations must recover the classical 1/5-power
turbulent flat-plate results $\theta = 0.036\,x\,Re_x^{-1/5}$ and
$C_f = 0.0592\,Re_x^{-1/5}$. Nothing in the closure is fitted to these, so it
is a genuine check.

In [ ]:
plate_viscosity = 1e-6
plate_speed = 1.0
plate_coordinate = collect(range(0.05, 5.0; length=400))
plate = solve_head_boundary_layer(
    plate_coordinate, fill(plate_speed, length(plate_coordinate)), plate_viscosity,
)
plate_Re = plate_speed .* plate_coordinate ./ plate_viscosity
plate_theta_law = 0.036 .* plate_coordinate .* plate_Re .^ (-0.2)
plate_cf_law = 0.0592 .* plate_Re .^ (-0.2)

for index in (50, 100, 200, 400)
    println((x=plate_coordinate[index], Re_x=plate_Re[index],
             theta=plate.momentum_thickness[index],
             theta_law=plate_theta_law[index],
             theta_ratio=plate.momentum_thickness[index] / plate_theta_law[index],
             H=plate.shape_factor[index],
             Cf=plate.skin_friction_coefficient[index],
             Cf_ratio=plate.skin_friction_coefficient[index] / plate_cf_law[index]))
end

The march sits a few per cent *below* both power laws and closes on them as
$Re_x$ grows — $\theta$ within $7\,\%$ at $Re_x=7\times10^{5}$ narrowing to
$3\,\%$ at $5\times10^{6}$, and $C_f$ from $9\,\%$ to $3\,\%$ over the same
range. That is the expected behaviour, not a defect: the 1/5-power laws are
themselves a fit valid for $5\times10^{5} \lesssim Re_x \lesssim 10^{7}$, and
the Ludwieg–Tillmann/Head combination is a different and more general closure
that happens to sit slightly under them. The shape factor settles to
$H\approx1.36$–$1.43$, the accepted flat-plate value.

## 2.4 The quasi-3D strip march, and the frictional drag it predicts

`solve_quasi3d_boundary_layer` runs the scalar march independently along every
longitudinal strip of the `StructuredPanelGrid`, using $|\boldsymbol{u}_e|$ for
the development and retaining all three components of $\boldsymbol{u}_e$ for the
direction of the wall shear,

$$\boldsymbol{\tau}_w = \tfrac12\rho\,C_f\,|\boldsymbol{u}_e|\,\boldsymbol{u}_e .$$

Strip width is approximated as $w_i = \Delta S_i / \Delta s_i$ so that the
displacement effect can be converted into an equivalent blowing velocity
through the surface — the **transpiration** boundary condition,

$$v_n^{BL} = \frac{1}{w}\frac{d}{ds}\big(w\,U_e\,\delta^{*}\big).$$

This is what makes the correction two-way capable: $v_n^{BL}$ is a Neumann
condition that can be handed straight back to the boundary-element solver, and
the resulting potential is an extra pressure field on the hull. With
`coupling_iterations > 0` the tangential velocity of that potential is fed back
into the edge condition under relaxation.

### What "quasi-3D" excludes

The geometry, edge velocity, shear direction, panel area and moment arm are all
three-dimensional, but the boundary-layer *development* along each strip is
scalar. There is no crossflow momentum-integral equation, no attachment-line
initialisation, and no wake continuation. This is the same fidelity class as
the on-body streamline method in the FlightStream theory manual, and the same
class as the classical Ikehata–Nagase–Maruo ship-stern method. For KVLCC2
specifically, the workshop measurements show a strong bilge vortex and
hook-shaped wake contours at $x/L_{pp}=0.9825$ that this model cannot
represent.

### Verification: total frictional drag

The right integrated check for an attached-flow method is the friction drag it
predicts at zero drift. Comparing against the ITTC-1957 model–ship correlation
line,

$$C_F = \frac{0.075}{(\log_{10}Re - 2)^2} = 3.450\times10^{-3}\quad\text{at }Re=4.6\times10^{6},$$

is the natural target. ITTC-57 is deliberately a few per cent *above* a true
flat-plate friction line, so a bare attached calculation with no form factor
should land slightly below it.

In [ ]:
friction_rows = map(((8, 5), (10, 6), (12, 7), (16, 9), (20, 12))) do shape
    grid = read_gothenburg2010_panel_grid(surface_paths; target_shape=shape)
    mesh = grid.mesh
    surge = solve_rigid_body_potential(mesh, :surge)
    edge_velocity = body_relative_edge_velocity(
        mesh, forward_speed, 0.0, 0.0;
        surge_gradient=surge.potential_gradient,
    )
    layer = solve_quasi3d_boundary_layer(
        grid, edge_velocity, kinematic_viscosity; rho=WATER_DENSITY,
    )
    area = mesh_surface_area(mesh)
    (; shape, panels=mesh.nfaces, strips=length(grid.strips), area,
       C_F=-layer.force[1] / (0.5 * WATER_DENSITY * forward_speed^2 * area),
       lateral_force=layer.force[2],
       separated=count(layer.separated),
       grid, layer, edge_velocity)
end

ittc_1957 = 0.075 / (log10(REYNOLDS_NUMBER) - 2)^2
println("ITTC-1957 C_F = $(ittc_1957)")
for row in friction_rows
    println((row.shape, row.panels, C_F=row.C_F, ratio=row.C_F / ittc_1957,
             separated=row.separated, lateral_force=row.lateral_force))
end

The predicted $C_F$ is $0.947$–$0.950$ times ITTC-57 and is essentially
**mesh-independent** across a sevenfold change in panel count — the closure,
the strip march, the wall-shear direction and the area weighting are all doing
the right thing. The lateral force is zero to machine precision at $v=r=0$, as
starboard–port symmetry requires. The number of separated panels grows from 0
to 14 as the stern is resolved, which is the model telling us where it stops
being valid.

The bottom row of the next figure shows the converged boundary-layer state on
a **developed surface**: each row is one girthwise strip, plotted at its mean
depth, and each column is a marching station, with the bow at the right. This
is the natural coordinate system for a marching method and, unlike a plan view,
it does not overlay the port and starboard halves. Colour ranges are clipped at
the 92nd percentile, because the last stern column is one to two orders of
magnitude above the rest of the hull and would otherwise wash it out — that
concentration at the stern is itself the result.

In [ ]:
detail = friction_rows[4]          # 16 × 9
detail_mesh = detail.grid.mesh
detail_layer = detail.layer

figure = Figure(size=(1750, 1000), backgroundcolor=:white)

plate_theta_axis = Axis(figure[1, 1];
    title="Flat plate: momentum thickness", xlabel="Re_x", ylabel="θ / x",
    xscale=log10, yscale=log10)
lines!(plate_theta_axis, plate_Re, plate_theta_law ./ plate_coordinate;
    color=:black, linestyle=:dash, linewidth=2.5, label="0.036 Re_x^(−1/5)")
lines!(plate_theta_axis, plate_Re, plate.momentum_thickness ./ plate_coordinate;
    color=:dodgerblue3, linewidth=2.5, label="Head march")
axislegend(plate_theta_axis; position=:rt)

plate_cf_axis = Axis(figure[1, 2];
    title="Flat plate: skin friction", xlabel="Re_x", ylabel="C_f",
    xscale=log10, yscale=log10)
lines!(plate_cf_axis, plate_Re, plate_cf_law;
    color=:black, linestyle=:dash, linewidth=2.5, label="0.0592 Re_x^(−1/5)")
lines!(plate_cf_axis, plate_Re, plate.skin_friction_coefficient;
    color=:dodgerblue3, linewidth=2.5, label="Head march")
axislegend(plate_cf_axis; position=:rt)

drag_axis = Axis(figure[1, 3];
    title="KVLCC2 friction drag vs ITTC-1957",
    xlabel="hull panels", ylabel="C_F", xscale=log10)
scatterlines!(drag_axis, [row.panels for row in friction_rows],
    [row.C_F for row in friction_rows];
    color=:seagreen4, linewidth=2.5, marker=:circle, markersize=11,
    label="quasi-3D Head march")
hlines!(drag_axis, [ittc_1957]; color=:black, linestyle=:dash, linewidth=2.5,
    label="ITTC-1957 correlation line")
ylims!(drag_axis, 0.0030, 0.0037)
axislegend(drag_axis; position=:rb)

# Developed-surface maps. The strips are the natural coordinate system for a
# marching method: one row per girthwise station line, ordered bow to stern.
strips = detail.grid.strips
station_count = length(first(strips))
all(length(strip) == station_count for strip in strips) ||
    error("expected a structured grid with equal-length strips")

developed(values) = [values[strips[strip][station]]
                     for station in 1:station_count, strip in eachindex(strips)]
station_x = [
    sum(detail_mesh.centers[strips[strip][station], 1] for strip in eachindex(strips)) /
    length(strips)
    for station in 1:station_count
]
# Label each strip by its mean depth rather than by its index, so the vertical
# axis of the developed view is physical.
strip_depth = [
    sum(detail_mesh.centers[panel, 3] for panel in strip) / length(strip)
    for strip in strips
]

"""Clip the colour range at a quantile so a few extreme stern panels do not
wash the rest of the hull out."""
function clipped_range(values; quantile_level=0.92, symmetric=false)
    sorted = sort(abs.(vec(values)))
    limit = sorted[max(1, round(Int, quantile_level * length(sorted)))]
    symmetric && return (-limit, limit)
    return (minimum(values), max(limit, minimum(values) + eps()))
end

function developed_map!(position, values, title, colormap; symmetric=false)
    matrix = developed(values)
    axis = Axis(figure[position...]; title=title,
        xlabel="x / Lpp   (bow at right)", ylabel="strip mean depth  z / Lpp")
    limits = clipped_range(matrix; symmetric)
    heatmap!(axis, station_x, strip_depth, matrix;
        colormap=colormap, colorrange=limits)
    Colorbar(figure[position[1] + 1, position[2]]; limits=limits,
        colormap=colormap, vertical=false, flipaxis=false, height=12)
    return axis
end

developed_map!((2, 1), detail_layer.displacement_thickness .* 1e3,
    "Displacement thickness δ* × 10³ / Lpp", :viridis)
shape_axis = developed_map!((2, 2), detail_layer.shape_factor,
    "Shape factor H   (× = separation flag, H ≥ 2.4)", :plasma)
separated_matrix = developed(detail_layer.separated)
separated_stations = [Point2f(station_x[i[1]], strip_depth[i[2]])
                      for i in findall(separated_matrix)]
isempty(separated_stations) ||
    scatter!(shape_axis, separated_stations; color=:cyan, marker=:xcross,
             markersize=13)
developed_map!((2, 3), detail_layer.transpiration_velocity,
    "Transpiration velocity v_n [m/s]", :balance; symmetric=true)

Label(figure[0, :],
    "Head integral boundary layer: closure verification and KVLCC2 surface state ($(detail_mesh.nfaces) panels)";
    fontsize=23, font=:bold)
rowsize!(figure.layout, 2, Relative(0.36))
save(joinpath(results_directory, "kvlcc2_boundary_layer.png"), figure; px_per_unit=1.5)
figure

## 2.5 From the boundary layer to derivative corrections

`viscous_maneuvering_correction` linearizes the whole quasi-3D solve about
$v = r = 0$. Its outputs are the transpiration field, the lateral shear force
and the shear yaw moment; differentiating those with respect to $(v, r)$ gives
two contributions to each velocity derivative:

**Shear.** $\partial F_y/\partial v$, $\partial F_y/\partial r$,
$\partial M_z/\partial v$, $\partial M_z/\partial r$ taken directly from the
integrated wall traction.

**Pressure.** The differentiated transpiration $\partial v_n^{BL}/\partial v_j$
is solved as a Neumann problem through the *same* boundary-element operator,
giving a correction potential $\psi_j$, and the resulting pressure is
integrated exactly as the inviscid convective term is:

$$\Delta Q_j^{\,p} = -\rho U\int_{S_h}\frac{\partial\psi_j}{\partial x}\,g_Q\,\mathrm{d}S .$$

The Jacobian is taken by forward-mode AD when the solve path is smooth, and by
central differences when it is not — `linearization = :auto` switches to
central differencing as soon as any panel is flagged separated (the separation
clamp is non-smooth) or coupling iterations are requested (the coupled solve
path is complex-valued). That switch is recorded in the result.

> **Do not combine this with the Schmitz truncation.** The truncation is
> *already* a semi-empirical viscous-and-vortex correction. Applying both
> counts the same physics twice. The rows below keep them separate.

The inviscid rows use the current defaults — the double-body linearization of
§1.6 and the direct-formulation acceleration derivatives. Wang's original
uniform-stream linearization is reported alongside so the size of that
modeling difference is visible on a real hull.

### Reference values

The comparison target is the model-test-derived MMG linear hull derivative set
for KVLCC2. MMG normalises forces by $\tfrac12\rho L T U^2$ where Wang uses
$\tfrac12\rho L^2 U^2$, so with $\lambda_T = T/L$ every velocity derivative
converts as $(\,\cdot\,)_W = \lambda_T\,(\,\cdot\,)_{MMG}$ —
`mmg_to_wang_velocity_derivatives`.

In [ ]:
reference = TOML.parsefile(joinpath(
    project_root, "validation", "kvlcc2_maneuvering", "reference.toml",
))["mmg"]["linear_hull"]
mmg_reference = mmg_to_wang_velocity_derivatives(
    MMGLinearHullDerivatives(reference["Y_v"], reference["Y_r"],
                             reference["N_v"], reference["N_r"]),
    DRAFT, LENGTH_PP,
)
velocity_fields(d) = (Y_v=d.Y_v, Y_r=d.Y_r, N_v=d.N_v, N_r=d.N_r)
reference_fields = velocity_fields(mmg_reference)

maximum_section = gothenburg_maximum_section(surface_paths)
println("Schmitz cut at x/Lpp = $(maximum_section.x)")
reference_fields

In [ ]:
prime(d) = velocity_fields(nondimensionalize_maneuvering_derivatives(
    d, 1.0, forward_speed; rho=WATER_DENSITY,
))
prime_viscous(d) = velocity_fields(nondimensionalize_viscous_derivatives(
    d, 1.0, forward_speed; rho=WATER_DENSITY,
))

derivative_rows = map(((8, 5), (10, 6), (12, 7), (16, 9))) do shape
    grid = read_gothenburg2010_panel_grid(surface_paths; target_shape=shape)
    mesh = grid.mesh

    inviscid = solve_potential_flow_maneuvering(
        mesh, forward_speed; rho=WATER_DENSITY,
    )
    surge = solve_rigid_body_potential(mesh, :surge)
    correction = viscous_maneuvering_correction(
        grid, surge, inviscid, forward_speed, kinematic_viscosity;
        rho=WATER_DENSITY,
    )
    corrected = apply_viscous_correction(inviscid.derivatives, correction)

    # Identical settings to the whole-hull row, so the stern cut is the only
    # difference between them.
    schmitz = solve_potential_flow_maneuvering(
        mesh, forward_speed; rho=WATER_DENSITY,
        velocity_mask=wang_stern_mask(mesh, maximum_section.x),
    )

    # Wang's original uniform-stream linearization, for comparison.
    wang_linearization = solve_potential_flow_maneuvering(
        mesh, forward_speed; rho=WATER_DENSITY,
        base_flow=:uniform_stream, acceleration_formulation=:indirect,
    )

    (; shape, panels=mesh.nfaces,
       whole_hull=prime(inviscid.derivatives),
       wang_linearization=prime(wang_linearization.derivatives),
       schmitz=prime(schmitz.derivatives),
       shear=prime_viscous(correction.shear_derivatives),
       pressure=prime_viscous(correction.pressure_derivatives),
       corrected=prime(corrected),
       separated=count(correction.base_boundary_layer.separated),
       linearization=correction.linearization)
end

for row in derivative_rows
    println((row.shape, row.panels, row.separated, row.linearization))
    println("   whole-hull, double-body linearization : $(row.whole_hull)")
    println("   whole-hull, Wang uniform-stream form  : $(row.wang_linearization)")
    println("   + Head viscous correction             : $(row.corrected)")
    println("   Schmitz truncated                     : $(row.schmitz)")
end
println("   MMG reference                         : $(reference_fields)")

In [ ]:
finest = last(derivative_rows)
coefficients = (:Y_v, :Y_r, :N_v, :N_r)
methods = (
    ("whole-hull inviscid", finest.whole_hull, :gray60),
    ("Wang linearization", finest.wang_linearization, :steelblue3),
    ("Schmitz truncated", finest.schmitz, :darkorange2),
    ("whole hull + Head", finest.corrected, :seagreen4),
)

figure = Figure(size=(1650, 620), backgroundcolor=:white)
for (index, coefficient) in enumerate(coefficients)
    labels = vcat([first(m) for m in methods], "MMG reference")
    axis = Axis(figure[1, index]; title=String(coefficient) * "′  (Wang scale)",
        xticks=(1:length(labels), labels), xticklabelrotation=pi / 5)
    for (position, (_, values, color)) in enumerate(methods)
        barplot!(axis, [position], [getproperty(values, coefficient)];
            color=color, width=0.6)
    end
    barplot!(axis, [length(methods) + 1], [getproperty(reference_fields, coefficient)];
        color=:black, width=0.6)
    hlines!(axis, [0.0]; color=:black, linewidth=1)
end

Label(figure[0, :],
    "KVLCC2 linear velocity derivatives, $(finest.panels) panels, against model-test MMG values";
    fontsize=23, font=:bold)
save(joinpath(results_directory, "kvlcc2_velocity_derivatives.png"), figure;
    px_per_unit=1.5)
figure

## 2.6 Interpretation

This comparison is a **validation failure in the scientific sense, and it is
reported as one.** No closure constant has been tuned to the four reference
coefficients, and none should be.

What the numbers say:

* The whole-hull inviscid $Y_v'$ is near zero and shrinking with refinement,
  exactly as notebook 1 predicts — ideal flow has no lateral damping.
* The Head correction supplies lateral damping of the right sign but roughly a
  fifth of the required magnitude. That is the expected outcome: an *attached*
  boundary layer cannot generate the cross-flow lift and vortex shedding that
  dominate $Y_v$ for a full-form hull at drift.
* $N_v'$ is the coefficient the potential flow gets closest on, because it is
  dominated by the Munk moment, which is an inviscid effect. It is also where
  the linearization matters most: the double-body form gives $-0.01484$ at 480
  panels against $-0.01725$ for Wang's uniform-stream form, a $14\,\%$ shift
  toward the reference $-0.008905$. The remaining gap is closed not by better
  inviscid modeling but by the stern truncation.
* $Y_r'$ has the wrong sign in every variant computed by direct truncated
  integration. Wang et al. anticipated this: $Y_r'$ and $N_r'$ need a
  truncation station *different* from the one used for $Y_v'$ and $N_v'$, and
  they recommend Clarke's regressions instead. Pass
  `rotational_velocity_mask` for a separate cut, or use
  `clarke_rotational_derivatives`, which for KVLCC2 gives $Y_r' = +0.00382$
  and $N_r' = -0.00255$ against the references $+0.005395$ and $-0.003185$ —
  right sign, right order, from the sway derivative alone.
* The mesh sequence is not converged, and the two finest meshes flag separated
  panels at the stern.

One result is worth stating plainly because it cuts against the method.
Correcting the linearization made the **Schmitz-truncated** agreement *worse*:
under Wang's uniform-stream form the 480-panel cut gave $Y_v'=-0.02180$ and
$N_v'=-0.00908$ against references of $-0.02048$ and $-0.00891$ — 6.5 % and
2.0 %. With the correct double-body linearization the same cut gives
$-0.01879$ and $-0.00778$, or 8.2 % and 12.6 %. The earlier agreement was
partly a cancellation between a modeling error and the error in a fitted stern
cut. That is a good reason to treat these truncated coefficients as calibrated
rather than predicted, and it is exactly the kind of thing that only shows up
when each piece is verified against something independent.

The physics that is missing is identifiable, not mysterious: surface-streamline
topology instead of fixed longitudinal strips, a crossflow momentum-integral
equation, and a stern wake carrying the momentum defect and the bilge vortex.
The Gothenburg 2010 measurements at $x/L_{pp} = 0.85,\ 0.9825,\ 1.1$ are the
right targets for that work, and they are recorded in
`validation/gothenburg2010/local_flow_reference.toml`.

What *is* validated here is the layer itself: the closure reproduces the
flat-plate laws, and the strip march predicts total frictional drag to within
5 % of ITTC-57 with no mesh sensitivity. The correction machinery is sound; the
flow model it is fed is not yet rich enough for the stern of a very full hull.